# P1 — Independent application circuit validation — corrected adapter

**Correction: relative branch offsets.** The previous adapter used named NetQASM labels, which the pinned Qoala interpreter treated as relative jumps after the parser resolved them to absolute indices. This skipped required server gates. This revision uses the correct relative offsets and checks all 48 parsed server branch paths before native simulation. The circuit, seeds, grid, noise model and density checks are unchanged.

**Keep the failed run and its ZIP.** Run this notebook with `RESUME_RUN_DIR="auto"`; its distinct source identity automatically creates a new run folder. Do not point it at the old failed folder. The calibration and timing-sensitivity notebooks need no change for this fix.

**Run All on the existing P1 server.** This notebook embeds all source and requires no installation. It repeats **336 exact NumPy branch controls**, then runs **672 native Qoala circuit fixtures** with fixed inputs and four fixed seeds per configuration. The native part is **pending server execution**; the accompanying local results establish only the NumPy controls.

The two-node programs implement the published delegated quantum computation circuit: a client measures its EPR half, sends the computed correction angle, and the server performs its dependent circuit and an adapted target measurement. Six angle settings, both Bell conventions, two EPR models and seven ages are fixed before execution. Exact controls enumerate both client outcomes; four native draws need not observe both outcomes in every cell.

**Scope:** an evaluator supplies the raw EPR density matrix and a phenomenological isotropic channel. `imposed_channel_age_ms` is a chosen noise parameter. `actual_simulated_lifetime_ns` is the independently recorded interval from Bob's EPR-request completion to the channel hook. These are different quantities. There is no artificial waiting block, hardware runtime reproduction, attacker, deadline selection, or Q16 calibration input.

Default **WORKERS=8** for the separately allocated eight-core machine. Worker count is adjustable without changing scientific identity or losing completed checkpoints from this corrected version. Each child process fixes BLAS/OpenMP threads to one. If sharing an allocation, keep the combined worker count within its available cores.


In [ ]:
from pathlib import Path
PROJECT_DIR = Path.home() / "P1"
WORKERS = 8
RESUME_RUN_DIR = "auto"  # Reuses the same frozen source; None starts a new run.
RUN_NATIVE = True       # False permits NumPy-only execution; native status remains pending.
PYTHON = None           # Existing P1 interpreter; no packages will be installed.
REPO = None             # Existing clean pinned Qoala checkout.
EXPORT_DIR = Path.cwd()


In [ ]:
import base64, fcntl, gzip, hashlib, json, os, sys
SOURCE_BUNDLE_SHA256 = '209ca05874faa916a4634888b73c959e84b859973fca8884574e1dee9f334863'
_EMBEDDED = '...'
_EXPECTED_FILES = ['adapter_revision.json', 'dqc_control_flow.py', 'dqc_local.py', 'dqc_model.py', 'dqc_native.py', 'dqc_programs.py', 'dqc_server.py', 'pinned_source_hashes.json', 'retained_physics_reference.py', 'source_provenance.json', 'test_control_flow.py', 'test_offline.py']

def install_embedded_code(project):
    data = gzip.decompress(base64.b64decode(_EMBEDDED))
    if hashlib.sha256(data).hexdigest() != SOURCE_BUNDLE_SHA256:
        raise RuntimeError("Embedded bundle checksum mismatch")
    files = json.loads(data)
    if set(files) != set(_EXPECTED_FILES):
        raise RuntimeError("Unexpected embedded files")
    base = Path(project).expanduser().resolve() / "experiment_code"
    base.mkdir(parents=True, exist_ok=True)
    folder = base / ("independent-dqc-" + SOURCE_BUNDLE_SHA256[:16])
    with (base / (folder.name + ".install.lock")).open("a+") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        folder.mkdir(exist_ok=True)
        for name, content in files.items():
            if Path(name).name != name:
                raise RuntimeError("Invalid embedded filename")
            target = folder / name
            expected = content.encode()
            if target.exists() and target.read_bytes() != expected:
                raise RuntimeError("Installed source changed: " + str(target))
            if not target.exists():
                temporary = target.with_name(name + ".tmp-" + str(os.getpid()))
                temporary.write_bytes(expected)
                os.replace(temporary, target)
    return folder

PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
EXPORT_DIR = Path(EXPORT_DIR).expanduser().resolve()
CODE_DIR = install_embedded_code(PROJECT_DIR)
print("Frozen source:", CODE_DIR)
print("Bundle SHA256:", SOURCE_BUNDLE_SHA256)


## Fixed model and completion gates

The settings are alpha in {pi/2, pi} and theta in {pi/4, pi/2, pi}. Storage ages are **0, 0.3, 1, 3, 4.8, 8.95 and 13 ms**. The channel contracts all Bloch components by `exp(-(age_ms/13)**1.67)`. This is the source's heuristic envelope, not a measured full isotropic process.

The source rounded Psi+ matrix has trace **1.001** and is normalized explicitly. The Psi- fixture is constructed by server Pauli-Z conjugation; it is not a second measured state. Psi+ correction is chronological Rx(pi); Psi- correction is Rx(pi), then Rz(pi). Every joint density uses (client, server) qubit order.

The adapter uses chronological server rotations Ry(pi/2), Rx(delta), Rx(pi). They are algebraically equivalent to Rz(delta), Ry(pi/2), Rx(pi), using `Ry(pi/2) Rz(delta) = Rx(delta) Ry(pi/2)`. This establishes ideal-unitary equivalence only; it does not establish equal gate durations or equal behavior on a noisy physical device. The fixed 13 ms, 1.67 envelope follows the final supplement/reference; the deposited exploratory fit also mentions a distinct 13.2 ms, 1.63 pair, which is not substituted here.

Native fixtures use two nodes, one qubit each, ideal gates of zero duration, 1,000 ns host and Qnos instruction costs, a perfect 1,000 ns EPR link, 1,000 ns one-way classical latency, and native memory noise off. These are evaluator fixtures, not inferred hardware costs. A raw source density is injected once at the first physical rotation. The channel is applied immediately before the target-basis rotation. No simulated wait realizes the selected age.

The adapted target test rotates the source output target to |0>; the ideal source target is -Y for alpha=pi/2 and -Z for alpha=pi. Its output bit is distinct from published raw SSRO bits, tomography processing, or SSRO-corrected fidelities.

The local controls check all 336 branches against the unchanged retained reference and an independently written chronological implementation, retain all density snapshots, and check absolute-age channel splitting. The native audit checks every observed branch, density, classical feed-forward, raw injection, gate chronology, backend CPU/QPU task intervals and instrumented service records. It does not impose a sampled two-branch coverage gate. Every attempt and failure is included in the final ZIP.


In [ ]:
import shutil, signal, subprocess, time, traceback, zipfile
from datetime import datetime, timezone
from IPython.display import FileLink, display

if not isinstance(WORKERS, int) or isinstance(WORKERS, bool) or not 1 <= WORKERS <= 8:
    raise ValueError("WORKERS must be an integer from 1 to 8 within the shared server budget")
available = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
if available and WORKERS > available:
    raise ValueError("WORKERS exceeds the available CPU count")
RUN_ENV = os.environ.copy()
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    RUN_ENV[name] = "1"
RUN_ENV["PYTHONUNBUFFERED"] = "1"
RUN_ENV["PYTHONDONTWRITEBYTECODE"] = "1"
RUN_ENV.pop("PYTHONOPTIMIZE", None)
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
POINTER = RESULTS_DIR / (".independent-dqc-latest-" + SOURCE_BUNDLE_SHA256[:16] + ".json")

def put(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp-" + str(os.getpid()))
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")
    os.replace(temporary, path)

def choose_run_directory():
    if RESUME_RUN_DIR == "auto" and POINTER.exists():
        old = json.loads(POINTER.read_text())
        if old["source_bundle_sha256"] != SOURCE_BUNDLE_SHA256:
            raise RuntimeError("Different resume bundle")
        root = Path(old["run_dir"]).expanduser().resolve()
        if not root.is_dir():
            raise RuntimeError("Saved run folder missing: " + str(root))
        return root
    if RESUME_RUN_DIR is None or RESUME_RUN_DIR == "auto":
        return RESULTS_DIR / ("independent-dqc-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
    root = Path(RESUME_RUN_DIR).expanduser().resolve()
    if not root.is_dir():
        raise RuntimeError("Explicit resume folder does not exist")
    return root

RUN_DIR = choose_run_directory()
if not RUN_DIR.name.startswith("independent-dqc-"):
    raise RuntimeError("Use an independent-dqc-* folder separate from all other experiments")
if RUN_DIR == CODE_DIR or RUN_DIR in CODE_DIR.parents or CODE_DIR in RUN_DIR.parents:
    raise RuntimeError("Run directory overlaps code")
if EXPORT_DIR == RUN_DIR or RUN_DIR in EXPORT_DIR.parents:
    raise RuntimeError("Export directory must be outside the run")
RUN_DIR.mkdir(parents=True, exist_ok=True)
manifest = dict(source_bundle_sha256=SOURCE_BUNDLE_SHA256)
if (RUN_DIR / "NOTEBOOK_PROTOCOL.json").exists() and json.loads((RUN_DIR / "NOTEBOOK_PROTOCOL.json").read_text()) != manifest:
    raise RuntimeError("Resume source bundle differs")
put(RUN_DIR / "NOTEBOOK_PROTOCOL.json", manifest)
put(POINTER, dict(run_dir=str(RUN_DIR), source_bundle_sha256=SOURCE_BUNDLE_SHA256))
(RUN_DIR / "source").mkdir(exist_ok=True)
for source in CODE_DIR.iterdir():
    if source.is_file() and source.name in _EXPECTED_FILES:
        target = RUN_DIR / "source" / source.name
        if target.exists() and target.read_bytes() != source.read_bytes():
            raise RuntimeError("Frozen run source differs")
        if not target.exists():
            shutil.copyfile(source, target)
attempt = str(time.time_ns())
put(RUN_DIR / ("launch-" + attempt + ".json"), dict(workers=WORKERS, run_native=RUN_NATIVE,
    thread_pool_limit=1, note="Operational concurrency; excluded from scientific protocol identity"))

def export_current_run():
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    with (RUN_DIR / "RUNNING.lock").open("a+") as lock:
        fcntl.flock(lock, fcntl.LOCK_SH | fcntl.LOCK_NB)
        paths = [p for p in sorted(RUN_DIR.rglob("*")) if p.is_file() and p.name != "RUNNING.lock"]
        inventory = {str(p.relative_to(RUN_DIR)):dict(bytes=p.stat().st_size,
            sha256=hashlib.sha256(p.read_bytes()).hexdigest()) for p in paths}
        destination = EXPORT_DIR / ("P1_Independent_DQC_" + RUN_DIR.name + "_full.zip")
        temporary = destination.with_suffix(".zip.tmp")
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
            for p in paths:
                z.write(p, str(Path(RUN_DIR.name) / p.relative_to(RUN_DIR)))
            z.writestr(str(Path(RUN_DIR.name) / "EXPORT_MANIFEST.json"), json.dumps(dict(files=inventory,
                full_raw_included=True, completion="COMPLETE.json" in inventory), sort_keys=True))
        os.replace(temporary, destination)
    display(FileLink(os.path.relpath(destination, Path.cwd())))
    print("Full raw ZIP:", destination, flush=True)
    return destination

def run_child(arguments, label):
    log = RUN_DIR / (label + "-" + attempt + ".log")
    with log.open("w") as handle:
        process = subprocess.Popen(arguments, cwd=CODE_DIR, env=RUN_ENV, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                handle.write(line); handle.flush()
            code = process.wait()
            if code:
                raise RuntimeError(label + " failed; exit code " + str(code))
        except BaseException:
            for sig in (signal.SIGINT, signal.SIGTERM, signal.SIGKILL):
                try:
                    os.killpg(process.pid, sig)
                except ProcessLookupError:
                    break
                try:
                    process.wait(timeout=10)
                    break
                except subprocess.TimeoutExpired:
                    pass
            raise
        finally:
            process.stdout.close()

failure = None
ARCHIVE = None
try:
    repo_candidates = ([Path(REPO).expanduser()] if REPO is not None else
        [PROJECT_DIR / name for name in ("qoala-sim", "p1-qoala", "qoala")] +
        [Path.home() / "p1-qoala", Path.home() / "qoala-sim"])
    if RUN_NATIVE:
        repos = []
        for candidate in repo_candidates:
            if candidate.is_dir():
                found = subprocess.run(["git", "-C", str(candidate), "rev-parse", "HEAD"], capture_output=True, text=True)
                if found.returncode == 0 and found.stdout.strip() == "31fa2c9699578d227736557f35cd01a134c4bbde":
                    repos.append(candidate.resolve())
        if not repos:
            raise RuntimeError("Set REPO to the existing pinned Qoala checkout. Checked: " + repr(repo_candidates))
        REPO = repos[0]
        if RUN_DIR == REPO or RUN_DIR in REPO.parents or REPO in RUN_DIR.parents:
            raise RuntimeError("Results overlap the simulator checkout")
    candidates = ([shutil.which(str(PYTHON)) or str(Path(PYTHON).expanduser().absolute())] if PYTHON is not None else
        [str(PROJECT_DIR / ".venv-p1/bin/python"), str(PROJECT_DIR / ".venv/bin/python"), sys.executable])
    chosen = None
    preflight_results = []
    for candidate in dict.fromkeys(candidates):
        if not Path(candidate).is_file():
            continue
        code = ("import json,sys; from dqc_server import check_environment; print(json.dumps(check_environment(sys.argv[1])))"
                if RUN_NATIVE else "import numpy; print(numpy.__version__)")
        completed = subprocess.run([candidate, "-c", code] + ([str(REPO)] if RUN_NATIVE else []),
            cwd=CODE_DIR, env=RUN_ENV, capture_output=True, text=True)
        preflight_results.append(dict(python=candidate,returncode=completed.returncode,stdout=completed.stdout,stderr=completed.stderr))
        put(RUN_DIR / ("preflight-" + attempt + ".json"), preflight_results)
        if completed.returncode == 0:
            chosen = candidate
            print("Interpreter:", chosen, "| native:", RUN_NATIVE, flush=True)
            break
    if chosen is None:
        raise RuntimeError("No compatible existing interpreter. Set PYTHON; see the exported preflight diagnostics.")
    PYTHON = chosen
    run_child([PYTHON, "-u", "-m", "unittest", "discover", "-s", str(CODE_DIR), "-p", "test_offline.py", "-v"], "offline-checks")
    run_child([PYTHON, "-u", str(CODE_DIR / "dqc_local.py"), "--output", str(RUN_DIR / "numpy")], "numpy-controls")
    if RUN_NATIVE:
        run_child([PYTHON, "-u", str(CODE_DIR / "dqc_server.py"), "run", "--repo", str(REPO),
            "--run-dir", str(RUN_DIR), "--workers", str(WORKERS)], "native-coordinator")
    else:
        print("NumPy controls complete. Native validation remains PENDING_SERVER_EXECUTION.", flush=True)
except BaseException as error:
    failure = error
    put(RUN_DIR / ("NOTEBOOK_FAILED-" + attempt + ".json"), dict(error=traceback.format_exc()))
finally:
    try:
        ARCHIVE = export_current_run()
    except BaseException as export_error:
        print("Export failed; preserve the entire run folder:", RUN_DIR, repr(export_error), flush=True)
        if failure is None:
            failure = export_error
if failure is not None:
    raise failure
print("Run directory:", RUN_DIR)
print("Resume with the same notebook and RESUME_RUN_DIR='auto'; WORKERS may change.")


## Reading the result

Return the **full ZIP**, including a diagnostic ZIP if any check fails. Success requires the declared records, matching source hashes and successful independent audits. A failure does not establish a quantum effect. Failed and interrupted attempts remain alongside later completed checkpoints; they are never silently replaced by another random seed.

`numpy/summary.json` reports the exact branch controls. `native_summary.json` exists only after all 672 native cases pass. `COMPLETE.json` refers to native fixture completion. The summary reports how many cells happened to sample both client outcomes, without treating that count as a completeness requirement. Every native raw file retains both programs, inputs, EPR injection, density snapshots, imposed age, observed native lifetime, quantum instructions, CPU/QPU traces and service records.

The simulator environment is Python 3.10, NumPy 1.26.4, NetSquid 1.1.8, pydynaa 1.0.2, Qoala 2.1.0, netqasm 2.3.0, at Qoala commit `31fa2c9699578d227736557f35cd01a134c4bbde`. The seven pinned source hashes, including the QNOS instruction processor, and clean tracked checkout are checked before native execution. The new `control_flow_preflight.json` records all 48 checked branch paths. Parser or interface incompatibility stops with diagnostics; it is not repaired by substituting another circuit.

After extracting a complete server ZIP, audit it from its run directory with a Python interpreter containing NumPy:

```bash
python source/dqc_server.py audit --run-dir .
```

Use `export_current_run()` to export again from this notebook. Rerun with `RESUME_RUN_DIR="auto"` to reuse verified native checkpoints. Changing WORKERS only changes concurrency. Changing any embedded source requires a distinct run identity.

Source: Nature 2025, *An operating system for executing applications on quantum network nodes*, DOI 10.1038/s41586-025-08704-w; Supplementary Information sections 4.1–4.4 and Appendix B, Figs. 17–18. The retained numerical oracle and inspected deposited simulation provide circuit/noise provenance. The ideal density controls neither reproduce the published SSRO analysis nor validate all physical timing parameters. Source code and supplementary descriptions do not establish a deployed tenant API or an attack.
